# Ravok Llama 3.2 1B QLoRA Fine-Tuning

This Google Colab notebook fine-tunes a small local-assistant model using synthetic examples grounded only in the approved passport reference. It does not use applicant PII or government credentials.

**Purpose:** improve concise factual answers and conservative abstention. **Not guaranteed:** this QLoRA run makes the model an official source or verifies current government rules.

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade --no-deps trl peft accelerate bitsandbytes datasets sentencepiece

The environment packages are installed. Colab should report a CUDA GPU; the next cell stops with a clear error if no usable GPU is available.

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. Runtime > Change runtime type > T4 GPU.")
print(torch.cuda.get_device_name(0))

The GPU check establishes whether fine-tuning is feasible. CUDA memory can vary on the free tier, so the notebook uses 4-bit QLoRA to reduce memory use.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

The base model is loaded in 4-bit and adapted with LoRA. This changes only adapter weights, while the base model remains available for comparison.

In [ ]:
approved_reference = """
You are a guidance assistant for a Zambia passport-service prototype.
Applicants must be bona fide Zambian nationals. Passports are issued in 32-page and 48-page formats.
Validity is 5 years for children and 10 years for adults. Applicants attend a Passport Office in person for interview and collection.
The standard requirement is a copy of the applicant's NRC.
Fees: ordinary 32-page ZMW 320; express 32-page ZMW 520; ordinary 48-page ZMW 520; express 48-page ZMW 720;
lost passport replacement 32-page ZMW 420; lost passport replacement 48-page ZMW 620.
Ordinary processing is 14 working days in Lusaka, Ndola, and Livingstone; 21 working days in Solwezi, Chipata, Kabwe, Mongu, Chinsali, Mansa, Kasama, and Choma.
Express processing is 5 working days but is temporarily unavailable.
Service-specific items include full passport data page, full passport last stamped page, GRZ receipt for the police report, expired passport data page,
proof of damage, Zambia Police Report, birth certificate, marriage certificate if married, adoption certificate, naturalization certificate,
bestowal certificate, citizenship registration certificate, previous passport copy for renewal, guardian consent if 18 or below,
shortened birth certificate, parents' NRC copies or passports for applicants under 16, and a recent passport-size photo.
Not every item applies to every application.
The reference gives no photo dimensions, recommender rules, office addresses or hours, or detailed applicability for every service.
Do not invent missing official requirements or support contacts.
Application process: after login open Start Application under passport services, read the introduction and eligibility, select district, below/above-16 category, and service location, then complete mandatory fields marked with * and use Next.
The form includes applicant/personal details, residential/postal address, 32/48-page setting, ordinary/express choice, civil-status details, physical description, schools, parents, next of kin, and recommender details.
Upload certified clear documents in supported formats and within the portal limits. Use GS-1 signing with the 4-digit PIN; first-time GSB users generate a certificate and PIN.
Review the invoice, complete an available card or mobile-money payment, wait for Paid, download the receipt, use Next, and then Finish to submit. Save the reference and monitor portal/email updates.
""".strip()

import random
random.seed(42)

template_groups = {
    "fee_32_ordinary": (
        "What is the fee for an ordinary 32-page passport?",
        "ZMW 320.",
        ["How much does an ordinary 32-page passport cost?", "Cost of ordinary 32-page passport?", "What amount is listed for the ordinary 32-page option?"]),
    "fee_32_express": (
        "How much is an express 32-page passport?",
        "ZMW 520. Express processing is temporarily unavailable.",
        ["What fee applies to express 32-page?", "Cost for 32-page express passport?", "Is the 32-page express fee ZMW 520?"]),
    "fee_48_ordinary": (
        "What is the ordinary 48-page passport fee?",
        "ZMW 520.",
        ["How much is an ordinary 48-page?", "Cost of ordinary 48-page passport?", "What does ordinary 48-page cost?"]),
    "fee_48_express": (
        "What fee applies to an express 48-page passport?",
        "ZMW 720. Express processing is temporarily unavailable.",
        ["How much is express 48-page?", "Cost for 48-page express service?", "Is express 48-page ZMW 720?"]),
    "lost_32": (
        "How much is lost passport replacement for 32 pages?",
        "ZMW 420.",
        ["Fee for replacing a lost 32-page passport?", "Lost 32-page replacement cost?", "What is the replacement fee for lost 32-page?"]),
    "lost_48": (
        "How much is lost passport replacement for 48 pages?",
        "ZMW 620.",
        ["Fee for replacing a lost 48-page passport?", "Lost 48-page replacement cost?", "What is the replacement fee for lost 48-page?"]),
    "processing_lusaka": (
        "How long is ordinary processing in Lusaka?",
        "14 working days.",
        ["Ordinary turnaround in Lusaka?", "How many working days in Lusaka?", "Processing time for Lusaka ordinary application?"]),
    "processing_ndola": (
        "How long is ordinary processing in Ndola?",
        "14 working days.",
        ["Ordinary turnaround in Ndola?", "How many working days in Ndola?", "Processing time for Ndola ordinary application?"]),
    "processing_livingstone": (
        "How long is ordinary processing in Livingstone?",
        "14 working days.",
        ["Ordinary turnaround in Livingstone?", "How many working days in Livingstone?", "Processing time for Livingstone ordinary application?"]),
    "processing_21": (
        "How long is ordinary processing in Kasama?",
        "21 working days.",
        ["Ordinary turnaround in Chipata?", "How many working days in Mongu?", "Processing time for Solwezi ordinary application?"]),
    "express": (
        "Is express processing available?",
        "Express processing is temporarily unavailable. The listed processing time is 5 working days when offered.",
        ["Can I use express now?", "Is fast-track currently offered?", "Is 5-day express service available?"]),
    "child_validity": (
        "What is passport validity for a child?",
        "5 years.",
        ["How long is a child's passport valid?", "Validity period for a child passport?", "How many years for children's passports?"]),
    "adult_validity": (
        "What is passport validity for an adult?",
        "10 years.",
        ["How long is an adult passport valid?", "Validity period for an adult passport?", "How many years for adult passports?"]),
    "eligibility": (
        "Who is eligible to apply?",
        "The source says an individual applicant must be a bona fide Zambian national.",
        ["Who can apply for this passport?", "Eligibility requirement?", "Must applicants be Zambian nationals?"]),
    "attendance": (
        "Must applicants attend in person?",
        "Yes. Applicants must attend a Passport Office in person for the interview and to collect the passport.",
        ["Do I need to appear personally?", "Is interview and collection in person?", "Must I go to the office myself?"]),
    "nrc_requirement": (
        "What is the standard document requirement?",
        "The source lists a copy of the applicant's NRC as the standard requirement.",
        ["Do I need to provide an NRC copy?", "What NRC document is required?", "What is the standard NRC requirement?"]),
    "photo_unknown": (
        "What are the passport photo dimensions?",
        "The supplied reference gives no photo dimensions. I cannot verify them; check current OPAS or MoHAIS instructions.",
        ["Give the photo size.", "What photo dimensions should I use?", "Does the reference specify photo specifications?"]),
    "office_unknown": (
        "What are the passport office opening hours?",
        "The supplied reference gives no office addresses or opening hours. I cannot verify them; check current OPAS or MoHAIS instructions.",
        ["When does the passport office open?", "What are office hours?", "Is the office open on Saturday?"]),
    "support_unknown": (
        "What is the official support phone number?",
        "I cannot verify an official support contact while offline. Use details published in OPAS or by official MoHAIS channels; I will not guess a phone number.",
        ["Give me the support hotline.", "What is the MoHAIS phone number?", "What support email should I use?"]),
    "payment_unknown": (
        "Can I pay with mobile money?",
        "The supplied reference does not specify payment methods. Check the payment options shown in OPAS.",
        ["How can I pay?", "Are debit cards accepted?", "Does the reference list payment options?"]),
    "reversion_unknown": (
        "Do I have to pay again after reversion?",
        "The supplied reference does not specify reversion or repayment rules. Review the officer comment in OPAS and confirm through official channels.",
        ["Should I repay after a reverted application?", "Is reversion free?", "Do I need a refund after reversion?"]),
    "officer_comment": (
        "Why was my application reverted? Officer comment: Please upload a clearer copy of your NRC.",
        "The officer comment says to upload a clearer copy of your NRC. I cannot confirm another reason from that comment.",
        ["Officer note: Please upload a clearer copy of my NRC. Why was I reverted?", "Explain this officer comment: upload a clearer NRC copy.", "The officer comment requested a clearer NRC copy. What other reason is there?"]),
    "documents": (
        "What documents are always required for every passport application?",
        "The source lists a copy of the applicant's NRC as the standard requirement. It also lists service-specific items, but says not every item applies to every application.",
        ["Give every required document.", "Which documents always apply?", "Does every listed document apply to everyone?"]),
}

import random
random.seed(42)
keys = list(template_groups)
random.shuffle(keys)
test_count = max(5, round(len(keys) * 0.20))
test_keys = set(keys[:test_count])
train_examples, test_examples = [], []
for key, (base_question, answer, variants) in template_groups.items():
    rows = [{"question": base_question, "answer": answer}] + [{"question": variant, "answer": answer} for variant in variants]
    (test_examples if key in test_keys else train_examples).extend(rows)
print("Training template groups:", len(template_groups) - len(test_keys), "Test template groups:", len(test_keys))
print("Training examples:", len(train_examples), "Held-out examples:", len(test_examples))
assert len(set(x["question"] for x in train_examples) & set(x["question"] for x in test_examples)) == 0

The dataset was split after all examples were generated, with a fixed random seed. The 20 percent held-out set is not used for training. Analysis should treat exact string overlap as a supporting metric, not legal verification.

In [ ]:
from datasets import Dataset

def format_example(example):
    messages = [
        {"role": "system", "content": approved_reference},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

train_dataset = Dataset.from_list(train_examples).map(format_example)
test_dataset = Dataset.from_list(test_examples).map(format_example)
print(train_dataset[0]["text"][:500])

The examples are converted to the model chat format. The next cell establishes a before-training baseline on the held-out test set.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_answer(question, max_tokens=160):
    messages = [
        {"role": "system", "content": approved_reference},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip()

def evaluate(records):
    rows = []
    score = 0
    unknown_terms = ("does not specify", "doesn't specify", "no photo", "no office", "cannot verify", "will not guess", "cannot confirm")
    for record in records:
        answer = generate_answer(record["question"])
        expected = record["answer"].lower()
        predicted = answer.lower()
        required_words = [w for w in ["320", "520", "620", "720", "14 working days", "21 working days", "5 years", "10 years", "32-page", "48-page", "temporarily unavailable", "nrc", "in person", "bona fide zambian"] if w in expected]
        fact_match = bool(required_words) and all(word in predicted for word in required_words)
        abstention_match = any(term in expected for term in unknown_terms) and any(term in predicted for term in unknown_terms)
        exact_match = expected in predicted
        passed = exact_match or fact_match or abstention_match
        score += int(passed)
        rows.append({"question": record["question"], "expected": record["answer"], "predicted": answer, "passed": bool(passed)})
    return score / len(records), rows

baseline_accuracy, baseline_rows = evaluate(test_examples)
print("Baseline held-out accuracy:", round(baseline_accuracy, 3))

The baseline results show whether the unadapted model already follows the reference. After analysis of failures, the LoRA training run is performed only on the training split.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 160,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)
trainer.train()

Training is complete if no CUDA out-of-memory or optimizer error occurs. The next cell repeats the same held-out questions and compares exact/fact/abstention performance.

In [ ]:
tuned_accuracy, tuned_rows = evaluate(test_examples)
print("Baseline held-out accuracy:", round(baseline_accuracy, 3))
print("Tuned held-out accuracy:", round(tuned_accuracy, 3))

for row in tuned_rows:
    print("PASS" if row["passed"] else "FAIL", "|", row["question"])
    print("Expected:", row["expected"])
    print("Predicted:", row["predicted"])
    print()

The same test set is used before and after training. Do not deploy a tuned model if held-out accuracy decreases or unsupported claims appear. A passing test must still be reviewed manually against official, current OPAS/MoHAIS information.

In [ ]:
model.save_pretrained("ravok_lora")
tokenizer.save_pretrained("ravok_lora")

model.save_pretrained_gguf("ravok_gguf", tokenizer, quantization_method="q4_k_m")
print("GGUF export complete. Download the generated file from the Colab Files panel.")

The notebook exports a Q4_K_M GGUF artifact. On the Windows project machine, place the file in a stable path and create a Modelfile such that the system instruction continues to enforce source-grounded abstention.

In [ ]:
print("Windows Ollama import steps:")
print("1. Download ravok_gguf/*.gguf to the Windows machine.")
print("2. Create a Modelfile containing: FROM C:\\path\\to\\ravok.gguf")
print("3. Run: ollama create ravok-finetuned:latest -f Modelfile")
print("4. Set OLLAMA_MODEL=ravok-finetuned:latest in OPAS-SmartGuide/.env.local")
print("5. Restart SmartGuide and retest /api/assistant/status plus held-out questions.")

## Final summary

This notebook follows a held-out evaluation design and exports a small LoRA-fine-tuned model for offline testing. It is a prototype experiment, not evidence that the model is legally accurate or like Gemini. Deployment requires manual review, a held-out pass threshold, confirmation that unsupported claims do not appear, and explicit review of the approved source before treating fees or availability as current.